In [1]:
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np

In [2]:
## load model
model=load_model('model.h5')

## load label_encoder_gender_pkl

with open('label_encode_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)

## load label one_hot_encoder

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)

## load one_hot_encode
with open('one_hot_encode.pkl','rb') as file:
    one_hot_encode_geo=pickle.load(file)

In [3]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 34,
    'Tenure': 3,
    'Balance': 40000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,  # <- changed
    'EstimatedSalary': 30000
}

In [4]:
input_df = pd.DataFrame([input_data])

# Encode Gender
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

# Encode Geography
geo_encoded = one_hot_encode_geo.transform(input_df[['Geography']])

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=one_hot_encode_geo.get_feature_names_out(['Geography'])
)

# Combine
input_df = pd.concat(
    [
        input_df.drop('Geography', axis=1).reset_index(drop=True),
        geo_encoded_df.reset_index(drop=True)
    ],
    axis=1
)

print(input_df)

   CreditScore  Gender  Age  Tenure  Balance  NumOfProducts  HasCrCard  \
0          600       1   34       3    40000              2          1   

   IsActiveMember  EstimatedSalary  Geography_France  Geography_Germany  \
0               1            30000               1.0                0.0   

   Geography_Spain  
0              0.0  


In [5]:
input_scaled=scaler.transform(input_df)

In [7]:
prediction=model.predict(input_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step


In [8]:
prediction_prob=prediction[0][0]

In [9]:
if prediction_prob>0.5:
    print("this customer is likely to churn")
else:
    print("this customer is likely not to chrun")

this customer is likely not to chrun
